#  **FOCS PROJECT | Nicolò Bachiorri**

1. It is mandatory to use GitHub for developing the project.
2. The project must be a jupyter notebook.
3. There is no restriction on the libraries that can be used, nor on  the Python version.
5. All questions on the project must be asked in the Discussion forum on the course website.
6. At most 3 students can be in each group. You must create the groups by yourself. You can use the Discussion forum to create the groups.
7. You do not have to send me the project before the discussion.
8. You do not have to prepare any slides for the discussion.
9. You can use AI tools, but you have to describe in the notebook how you have used such tools and you have to show that you have fully understood everything that you have in your project.



# **TASKS**



1. Create a single dataframe with the concatenation of all input csv files, adding a column called country
2. Extract all videos that have no tag.
3. For each channel, determine the total number of views
4. Save all rows with disabled comments and disabled ratings, or that have video_error_or_removed in a new dataframe called excluded, and remove those rows from the original dataframe.
5. Add a like_ratio column storing the ratio between the number of likes and of dislikes
6. Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)
7. For each interval, determine the number of videos, average number of likes and of dislikes.
8. For each tag, determine the number of videos
Notice that tags contains a string with several tags.

9. Find the tags with the largest number of videos
10. For each (tag, country) pair, compute average ratio likes/dislikes
11. For each (trending_date, country) pair, the video with the largest number of views
12. Divide trending_date into three columns: year, month, day
13. For each (month, country) pair, the video with the largest number of views
14. Read all json files with the video categories
15. For each country, determine how many videos have a category that is not assignable.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import zstandard as zstd
import os


In [ ]:
### READ AND CONCAT THE CSV DATAFRAMES

countries = ["CA", "DE", "FR", "GB", "IN", "JP", "KR", "MX", "RU", "US"]
base_path = "./trendingYT/"

all_dfs = []

for country_code in countries:
    file_name = f"{country_code}videos.csv.zst"
    file_path = os.path.join(base_path, file_name)

    try:
        # Pandas can directly read .zst files, specifying 'latin1' encoding for broader compatibility
        if country_code in ["JP" , "KR" ,"MX","RU"]:
            df_country = pd.read_csv(file_path, encoding='latin1')
        else:
            df_country = pd.read_csv(file_path, encoding='UTF-8')

        df_country['country'] = country_code # Add a country column
        all_dfs.append(df_country)
        print(f"Successfully loaded {file_name}")


    except FileNotFoundError:
        print(f"File not found for {country_code}: {file_path}")
    except Exception as e:
        print(f"Error loading {file_name}: {e}")

# Concatenate all DataFrames into a single one
df = pd.concat(all_dfs, ignore_index=True)

display(df.head())
print(f"Total rows in combined DataFrame: {len(df)}")

Successfully loaded CAvideos.csv.zst
Successfully loaded DEvideos.csv.zst
Successfully loaded FRvideos.csv.zst
Successfully loaded GBvideos.csv.zst
Successfully loaded INvideos.csv.zst
Successfully loaded JPvideos.csv.zst
Successfully loaded KRvideos.csv.zst
Successfully loaded MXvideos.csv.zst
Successfully loaded RUvideos.csv.zst
Successfully loaded USvideos.csv.zst


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,CA
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,CA


Total rows in combined DataFrame: 375942


In [13]:
### EXTRACT VIDEOS THAT HAVE NO TAG
df.drop_duplicates(inplace = True)

#i asked to Claude.ai to provide me a list of usual string values used to indicate null vaulues
null_values = ['none', '[none]', 'null', 'nan', 'na', 'n/a', 'missing', 'unknown', '-', '?', '']
#i added '[none]' after df exploration
df['tags'] = df['tags'].str.lower()
df['tags'] = df['tags'].str.strip() 
df['tags'] = df['tags'].replace(null_values, np.nan)


no_tag = df[df['tags'].isnull()] 

display(no_tag.head()) 
print(f"Number of videos with no tags: {len(no_tag)}")

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
41,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,NaN,36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
58,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,NaN,106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
78,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,NaN,320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
86,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,NaN,116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA
98,lHcXhBojpeQ,17.14.11,三屆TVB視帝，拋棄10年青梅竹馬髮妻，為娶小三還不惜與母絕交！,明星百曉生,22,2017-11-12T12:49:50.000Z,NaN,88061,47,58,17,https://i.ytimg.com/vi/lHcXhBojpeQ/default.jpg,False,False,False,NaN,CA


Number of videos with no tags: 36221


In [14]:
### For each channel, determine the total number of views
channel_views = df.groupby('channel_title')['views'].sum().reset_index(name = 'total_views')
channel_views = channel_views.sort_values(by = 'total_views', ascending = False)
display(channel_views)

,channel_title,total_views
4590,ChildishGambinoVEVO,10900185104
15599,Marvel Entertainment,10120133557
17793,NickyJamTV,9479859505
18533,Ozuna,8623329509
28519,ibighit,7644304297
...,...,...
17503,NavylittleMonster,365
25931,Videostendencias,302
17904,No Comment TV,284
22639,Sport Life,163


In [15]:
### Save all rows with disabled comments and disabled ratings
### or that have video_error_or_removed in a new dataframe called
### excluded, and remove those rows from the original dataframe.

print(f"Shape of df BEFORE exclusion: {df.shape}")

condition = ((df['comments_disabled'] == True) & (df['ratings_disabled'] == True)) | (df['video_error_or_removed'] == True)
excluded = df[condition]
df = df[~condition]

print(f"Shape of 'excluded' DataFrame: {excluded.shape}")
print(f"Shape of 'df' AFTER exclusion: {df.shape}")

Shape of df BEFORE exclusion: (363372, 17)
Shape of 'excluded' DataFrame: (2438, 17)
Shape of 'df' AFTER exclusion: (360934, 17)


In [16]:
### Add a like_ratio column storing the ratio between
### the number of likes and of dislikes

#trying not to explode the ratio
epsilon = 1e-10
df['like_ratio'] = df['likes'] / (df['dislikes'] + epsilon)

In [17]:
### Cluster the publish time into 10-minute intervals (e.g. from 02:20 to 02:30)

df.columns
df['publish_time'] = pd.to_datetime(df['publish_time'])

#the floor function rounds to the lower cluster, e.g. 10:37 --> 10:30
df['publish_interval_10min'] = df['publish_time'].dt.floor('10min')

display(df[['publish_time', 'publish_interval_10min']].head())

,publish_time,publish_interval_10min
0,2017-11-10 17:00:03+00:00,2017-11-10 17:00:00+00:00
1,2017-11-13 17:00:00+00:00,2017-11-13 17:00:00+00:00
2,2017-11-12 19:05:24+00:00,2017-11-12 19:00:00+00:00
3,2017-11-12 18:01:41+00:00,2017-11-12 18:00:00+00:00
4,2017-11-09 11:04:14+00:00,2017-11-09 11:00:00+00:00


In [18]:
### For each tag, determine the number of videos
### Notice that tags contains a string with several tags.

exploded = df.copy()
exploded['tags'] = exploded['tags'].str.lower().str.split('|')
exploded = exploded.explode('tags')
exploded['tags'].value_counts()

tags
"funny"                            16562
"comedy"                           14688
"2018"                             10873
"news"                              8277
"music"                             7746
                                   ...  
"ssc je"                               1
"ies"                                  1
"general awareness"                    1
"current affairs 2018 in hindi"        1
"trying crayola crayons makeup"        1
Name: count, Length: 830918, dtype: int64

In [19]:
### Find the tags with the largest number of videos
# FUNNY 

display(exploded['tags'].value_counts().head())


tags
"funny"     16562
"comedy"    14688
"2018"      10873
"news"       8277
"music"      7746
Name: count, dtype: int64

In [20]:
### For each (tag, country) pair, compute average ratio likes/dislikes

condition = ((df['tags'].isnull()) | (df['tags'].str.contains("none")) | (df['tags'] == "" ))

exploded = exploded[~condition]
tag_like_ratio =  exploded.groupby(['tags', 'country'])['like_ratio'].mean().reset_index(name = 'avg_like_ratio')
display(tag_like_ratio.head())
print("VALORI ORDINATI PER TOP 5 AVG_LIKE_RATIO: ")
tag_like_ratio.sort_values(by = "avg_like_ratio", ascending = False).head(5)


/var/folders/y3/k9ydcjbs78730dhx2b2vrcdm0000gn/T/ipykernel_56535/2843245510.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  exploded = exploded[~condition]


,tags,country,avg_like_ratio
0,,CA,8.618739
1,,DE,24.344225
2,,FR,16.424636
3,,IN,6.977710
4,,JP,9.534380


VALORI ORDINATI PER TOP 5 AVG_LIKE_RATIO: 


,tags,country,avg_like_ratio
710569,"""wu yi fan""",GB,5.916691e+14
386323,"""likethat""",GB,5.916691e+14
386293,"""like that""",GB,5.916691e+14
973194,"""吴亦凡""",GB,5.916691e+14
367096,"""kw""",GB,5.916691e+14


In [21]:
df['trending_date'].value_counts() #le date hanno formato anno-giorno-mese

trending_date
18.12.02    1864
18.17.02    1861
18.06.03    1859
18.18.04    1855
18.20.02    1854
            ... 
18.02.02    1573
18.19.05    1531
18.15.03    1506
18.14.05    1474
18.20.05    1464
Name: count, Length: 205, dtype: int64

In [26]:
### For each (trending_date, country) pair, the video with the largest number of views 


df['trending_date'] = pd.to_datetime(df['trending_date'], format = "%y.%d.%m" , errors='coerce' ) 

idx = df.groupby(['trending_date', 'country'])['views'].idxmax()
result = df.loc[idx, ['trending_date', 'country', 'video_id', 'views']]

result = result.reset_index(drop=True)
result

,trending_date,country,video_id,views
0,2017-11-14,CA,2Vv-BfVoq4g,33523622
1,2017-11-14,DE,2Vv-BfVoq4g,33523622
2,2017-11-14,FR,2Vv-BfVoq4g,33523622
3,2017-11-14,GB,2Vv-BfVoq4g,33523622
4,2017-11-14,IN,ePO5M5DE01I,35885754
...,...,...,...,...
1962,2018-06-14,JP,#NAME?,4427381
1963,2018-06-14,KR,#NAME?,4427381
1964,2018-06-14,MX,gPHVLxm8U-0,5829270
1965,2018-06-14,RU,i63jWjoAWHE,6597033


In [27]:
### Divide trending_date into three columns: year, month, day

df['year'] = df['trending_date'].dt.year 
df['month'] = df['trending_date'].dt.month 
df['day'] = df['trending_date'].dt.day 

df.head(2)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_interval_10min,year,month,day
0,n1WpP7iowLc,2017-11-14,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10 17:00:03+00:00,"eminem|""walk""|""on""|""water""|""aftermath/shady/in...",17158579,787425,43420,...,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,CA,18.135076,2017-11-10 17:00:00+00:00,2017,11,14
1,0dBIkQ4Mz1M,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13 17:00:00+00:00,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,...,False,False,False,STill got a lot of packages. Probably will las...,CA,75.707346,2017-11-13 17:00:00+00:00,2017,11,14


In [28]:
### For each (month, country) pair, the video with the largest number of views

idx = df.groupby(['month', 'country'])['views'].idxmax()
result = df.loc[idx, ['month', 'country', 'video_id', 'views']]
result = result.reset_index(drop = True)
result


,month,country,video_id,views
0,1,CA,LsoLEjrDogU,43067983
1,1,DE,LsoLEjrDogU,37728802
2,1,FR,LsoLEjrDogU,37728802
3,1,GB,LsoLEjrDogU,90598955
4,1,IN,dfnCAmr569k,42019590
...,...,...,...,...
72,12,IN,FlsCjmMhFmw,125432237
73,12,KR,FlsCjmMhFmw,113876217
74,12,MX,FlsCjmMhFmw,100912384
75,12,RU,FlsCjmMhFmw,52611730


In [74]:
### **Structure of JSON file** 

#{
# "kind": "youtube#videoCategoryListResponse",
# "etag": "\"ld9biNPKjAjgjV7EZ4EKeEGrhao/1v2mrzYSYG6onNLt2qTj13hkQZk\"",
# "items": [
#  {
#   "kind": "youtube#videoCategory",
#   "etag": "\"ld9biNPKjAjgjV7EZ4EKeEGrhao/Xy1mB4_yLrHy_BmKmPBggty2mZQ\"",
#   "id": "1",
#   "snippet": {
#    "channelId": "UCBR8-60-B28hp2BmDPdntcQ",
#    "title": "Film & Animation",
#    "assignable": true
#   }



In [29]:
### Read all json files with the video categories 

import json 

countries = ["CA", "DE", "FR", "GB", "IN", "JP", "KR", "MX", "RU", "US"]
base_path = "./trendingYT/"

all_dfs = []

for country in countries: 

    json_name = f"{country}_category_id.json" 
    json_path = os.path.join(base_path , json_name) 

    df_cat = pd.json_normalize(json.load(open(json_path))['items']) 
    df_cat['country'] = country
    all_dfs.append(df_cat) 

cat = pd.concat(all_dfs , ignore_index= True)
print("Head of cat: ")
display(cat.head())
print("\n ")
print("Tail of cat: ")
display(cat.tail())
print(f"Shape of cat: {cat.shape}")


Head of cat: 


,kind,etag,id,snippet.channelId,snippet.title,snippet.assignable,country
0,youtube#videoCategory,"""ld9biNPKjAjgjV7EZ4EKeEGrhao/Xy1mB4_yLrHy_BmKm...",1,UCBR8-60-B28hp2BmDPdntcQ,Film & Animation,True,CA
1,youtube#videoCategory,"""ld9biNPKjAjgjV7EZ4EKeEGrhao/UZ1oLIIz2dxIhO45Z...",2,UCBR8-60-B28hp2BmDPdntcQ,Autos & Vehicles,True,CA
2,youtube#videoCategory,"""ld9biNPKjAjgjV7EZ4EKeEGrhao/nqRIq97-xe5XRZTxb...",10,UCBR8-60-B28hp2BmDPdntcQ,Music,True,CA
3,youtube#videoCategory,"""ld9biNPKjAjgjV7EZ4EKeEGrhao/HwXKamM1Q20q9BN-o...",15,UCBR8-60-B28hp2BmDPdntcQ,Pets & Animals,True,CA
4,youtube#videoCategory,"""ld9biNPKjAjgjV7EZ4EKeEGrhao/9GQMSRjrZdHeb1OEM...",17,UCBR8-60-B28hp2BmDPdntcQ,Sports,True,CA



 
Tail of cat: 


,kind,etag,id,snippet.channelId,snippet.title,snippet.assignable,country
306,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/N1TrDFLRppxZgBowC...",40,UCBR8-60-B28hp2BmDPdntcQ,Sci-Fi/Fantasy,False,US
307,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/7UMGi6zRySqXopr_r...",41,UCBR8-60-B28hp2BmDPdntcQ,Thriller,False,US
308,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/RScXhi324h8usyIet...",42,UCBR8-60-B28hp2BmDPdntcQ,Shorts,False,US
309,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/0n9MJVCDLpA8q7aiG...",43,UCBR8-60-B28hp2BmDPdntcQ,Shows,False,US
310,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/x5NxSf5fz8hn4loSN...",44,UCBR8-60-B28hp2BmDPdntcQ,Trailers,False,US


Shape of cat: (311, 7)


In [98]:
cat['snippet.assignable'].unique() #True/False

### left join tra df e cat 

cat.head() # only: 'id', 'snippet.title' , 'snippet.assignable' , 'country'
           # drop: 'kind' ,'etag' , 'snippet.channelId'

cat.drop(columns = ['kind' ,'etag' , 'snippet.channelId' ], axis = 1 , inplace = True) 
cat.rename(columns = {'id' : 'category_id'}, inplace = True)
cat.head() 

### left join df and cat
df['category_id'] = df['category_id'].astype(str)
merged_df = pd.merge(df , cat , on = ['category_id' , 'country'] , how = 'left') 
merged_df.head(2)

### For each country, determine how many videos have a category that is not assignable. 
not_assignable = merged_df[(merged_df["snippet.assignable"] == False)] 
not_assignable.groupby('country')['snippet.assignable'].count()

country
CA    130
DE    110
FR    112
GB     20
IN    159
KR    159
MX      3
RU    194
US     57
Name: snippet.assignable, dtype: int64